# Mejoras King County — Guía de Examen

## Tipo de problema
**Regresión supervisada** con datos **temporales** — predecir `price` (precio de venta de casas).

## Por qué King County es especial respecto a California
California Housing: casas de un año concreto, el tiempo no importa.
King County: ventas reales con fecha, el tiempo SÍ importa porque los precios suben con el tiempo.

**El error clásico**: hacer `train_test_split` aleatorio en datos temporales.
Si el test contiene ventas de enero y el train tiene ventas de diciembre del mismo año,
el modelo "verá el futuro" durante el entrenamiento → métricas de test infladas.

## Mejoras clave
1. **Split temporal** (80% ventas antiguas → 20% ventas recientes)
2. **Feature engineering** temporal: `house_age`, `was_renovated`, `sale_month`
3. **Ratios útiles**: `living_lot_ratio`, `bathrooms_per_bedroom`
4. **`TimeSeriesSplit`** en CV para respetar el tiempo también durante la búsqueda
5. **Eliminar `id`** del dataset (memoriza casas concretas, no generaliza)


## Paso 1 — Cargar datos y ordenar por fecha

**Por qué ordenar por fecha antes de cualquier cosa?**
Necesitamos que el orden de las filas sea cronológico para poder hacer el split temporal correctamente.
Si las filas están desordenadas y cortamos en el índice 80%, estaríamos mezclando fechas.

**Por qué convertir `date` a `datetime`?**
Pandas trata las fechas como strings si no convertimos. Con `datetime` podemos extraer año, mes,
ordenar cronológicamente y calcular duraciones.

**Por qué eliminar `id`?**
El `id` identifica una casa concreta. Si el modelo aprende que "el id 7129300520 vale $500k",
no está aprendiendo NADA útil. Memoriza datos de train y falla en test.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Ruta con fallback: primero busca en datasets/ local, luego en el repo
DATA_PATH = Path("datasets/king-county/kc_house_data.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../proyects-upgrades/datasets/king-county/kc_house_data.csv")
if not DATA_PATH.exists():
    # Último recurso: descargar con kagglehub
    import kagglehub
    path = kagglehub.dataset_download("harlfoxem/housesalesprediction")
    DATA_PATH = Path(path) / "kc_house_data.csv"

df = pd.read_csv(DATA_PATH)
print(f"Shape original: {df.shape}")
print(df.dtypes[["date", "price", "bedrooms", "id"]].to_string())


In [ ]:
# Convertir fecha y ordenar cronológicamente
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Eliminar id: no aporta información generalizables
df = df.drop(columns=["id"])

print(f"Rango de fechas: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Total ventas: {len(df):,}")


## Paso 2 — Feature Engineering

**Por qué `house_age` en lugar de `yr_built`?**
`yr_built = 1980` por sí solo no dice mucho. ¿Era nueva en 1980? ¿O antigua en 2014?
`house_age = año_venta - yr_built` da la antigüedad real en el momento de la venta.

**Por qué `was_renovated` y `years_since_renovation`?**
Una casa renovada recientemente vale más. Pero un número como `yr_renovated = 2003`
no es tan informativo como "fue renovada hace 10 años". Separamos en:
- `was_renovated`: variable binaria (fue renovada o no)
- `years_since_renovation`: si fue renovada, hace cuántos años

**Por qué `sale_month`?**
Los precios de casas tienen estacionalidad: más ventas (y precios más altos) en primavera/verano.

**Por qué `living_lot_ratio`?**
El tamaño del lote y el de la vivienda por separado dicen poco. Su ratio indica eficiencia de construcción.

**Por qué `bathrooms_per_bedroom`?**
Normaliza la cantidad de baños por el tamaño de la casa. Casas de lujo tienen más baños por habitación.


In [ ]:
# Extraer features temporales
df["sale_year"]  = df["date"].dt.year
df["sale_month"] = df["date"].dt.month

# Antigüedad real en el momento de la venta
df["house_age"] = df["sale_year"] - df["yr_built"]

# Features de renovación
df["was_renovated"]          = (df["yr_renovated"] > 0).astype(int)
df["years_since_renovation"] = np.where(
    df["yr_renovated"] > 0,
    df["sale_year"] - df["yr_renovated"],
    0    # 0 si nunca fue renovada
)

# Ratios
df["living_lot_ratio"]       = df["sqft_living"] / df["sqft_lot"]
df["bathrooms_per_bedroom"]  = df["bathrooms"] / df["bedrooms"].replace(0, np.nan)

# Eliminar columnas que ya están representadas por features derivadas
df = df.drop(columns=["date"])  # ya extrajimos year y month

print("Nuevas features:")
print(df[["house_age", "was_renovated", "years_since_renovation",
          "living_lot_ratio", "bathrooms_per_bedroom"]].describe().round(2))


## Paso 3 — Split Temporal (NO aleatorio)

**Por qué 80/20 por posición y no por `train_test_split`?**
`train_test_split` mezcla aleatoriamente. En datos temporales queremos:
- **Train**: ventas antiguas (el modelo aprende el pasado)
- **Test**: ventas recientes (evaluamos si predice el futuro)

`df.iloc[:split_idx]` toma las primeras 80% filas → las más antiguas.
`df.iloc[split_idx:]` toma las últimas 20% filas → las más recientes.

**¿Qué pasa si hacemos split aleatorio?**
El modelo puede ver en train una casa vendida en dic-2014 y en test la misma zona en ene-2014.
Como los precios suben con el tiempo, el modelo "sabe" el precio futuro → overfitting temporal.


In [ ]:
split_idx = int(len(df) * 0.8)

train_set = df.iloc[:split_idx]
test_set  = df.iloc[split_idx:]

X_train = train_set.drop(columns=["price"])
y_train = train_set["price"]
X_test  = test_set.drop(columns=["price"])
y_test  = test_set["price"]

print(f"Train: {len(X_train):,} ventas  ({X_train['sale_year'].min()}–{X_train['sale_year'].max()})")
print(f"Test:  {len(X_test):,} ventas   ({X_test['sale_year'].min()}–{X_test['sale_year'].max()})")
print("✓ El test solo contiene fechas POSTERIORES al train")


## Paso 4 — Pipeline y GridSearch con TimeSeriesSplit

**Por qué `TimeSeriesSplit` en lugar de `KFold` para la CV?**
`KFold` divide aleatoriamente → en cada fold el validación puede ser anterior al train.
`TimeSeriesSplit` garantiza que el fold de validación siempre es POSTERIOR al fold de train.
Así respetamos el orden temporal también dentro de la búsqueda de hiperparámetros.

```
TimeSeriesSplit(n_splits=3):
  Fold 1: Train=[0..33%]  Val=[33..50%]
  Fold 2: Train=[0..50%]  Val=[50..67%]
  Fold 3: Train=[0..67%]  Val=[67..80%]
```

**Por qué `GridSearchCV` aquí en vez de `RandomizedSearchCV`?**
Con un espacio pequeño de parámetros (2×3×3 = 18 combinaciones) es más rápido probarlas todas.
Si el espacio crece, cambia a `RandomizedSearchCV`.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols     = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print(f"Features numéricas: {len(numeric_cols)}")
print(f"Features categóricas: {categorical_cols}")


In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_cols),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor",    RandomForestRegressor(random_state=42, n_jobs=-1)),
])

param_grid = {
    "regressor__n_estimators":     [100, 200],
    "regressor__max_depth":        [12, 20, None],
    "regressor__min_samples_leaf": [1, 3, 5],
}

# TimeSeriesSplit: la validación siempre es posterior al entrenamiento
tscv = TimeSeriesSplit(n_splits=3)

search = GridSearchCV(
    model,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

search.fit(X_train, y_train)
print("Mejores parámetros:", search.best_params_)
print(f"Mejor RMSE en CV: {-search.best_score_:,.0f} $")


In [ ]:
y_pred = search.best_estimator_.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("=" * 40)
print("EVALUACIÓN FINAL EN TEST SET (FUTURO)")
print("=" * 40)
print(f"RMSE: {rmse:>12,.0f} $")
print(f"MAE:  {mae:>12,.0f} $")
print(f"R²:   {r2:>12.4f}")


## Explicación para el examen

> *"King County tiene datos temporales. El error clásico es hacer split aleatorio,
> que filtra información del futuro al entrenamiento (data leakage temporal).
> Lo correo es ordenar por fecha y poner el 80% más antiguo en train y el 20% más reciente en test.
> Del mismo modo, uso TimeSeriesSplit en la búsqueda de hiperparámetros.
> El feature engineering explota la fecha para extraer antigüedad de la casa y estacionalidad."*

## Problemas frecuentes y soluciones

| Problema | Causa | Solución |
|----------|-------|---------|
| `FileNotFoundError` | CSV no descargado | Descargar antes del examen con `kagglehub` |
| RMSE muy bajo sospechoso | Split aleatorio con leakage | Cambiar a split temporal |
| `TimeSeriesSplit` da error | No importado | `from sklearn.model_selection import TimeSeriesSplit` |
| NaN en `bathrooms_per_bedroom` | División por 0 (casas sin habitaciones) | Usar `.replace(0, np.nan)` antes de dividir |
| Features temporales no ayudan | `date` incluida como string | Convertir a datetime y extraer year/month |
